# 09. Advanced RAG: Reranking & Context Compression

## 학습 목표

- 검색된 문서를 그대로 사용하지 않고 재정렬하거나 압축한다.
- 답변에 필요한 핵심 근거만 정리한다.

## 공통 전제

- 실습 데이터: `../data/공직자_민원응대_핵심_매뉴얼.pdf`
- 기본 모델: `gpt-4o-mini`
- 기본 임베딩 모델: `text-embedding-3-small`
- 벡터DB: Qdrant 로컬 인메모리 모드

> 이 노트북은 `src` 파일을 import하지 않고, 노트북 안의 코드만으로 실행되도록 구성한다.

In [3]:
from pathlib import Path
import re
from dotenv import load_dotenv
from pypdf import PdfReader

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_qdrant import QdrantVectorStore
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv(override=True, dotenv_path="../../.env")

True

## 1. 검색 결과 가져오기

In [5]:
from langchain_openai import OpenAIEmbeddings
from langchain_qdrant import QdrantVectorStore
QDRANT_URL = "http://localhost:6333"
COLLECTION_NAME = "civil_complaint_manual_medium"

EMBEDDING_MODEL = "text-embedding-3-small"

# True: collection을 삭제 후 새로 생성한다.
# False: 이미 저장된 collection을 재사용한다.
RECREATE_COLLECTION = True

embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)

vector_store = QdrantVectorStore.from_existing_collection(
    embedding=embeddings,
    collection_name=COLLECTION_NAME,
    url=QDRANT_URL,
)

In [6]:
question = "권장시간이 지났는데도 계속 상담을 요구하면 어떻게 하나요?"

docs = vector_store.similarity_search(question, k=6)

for i, doc in enumerate(docs, start=1):
    print(f"--- 검색 결과 {i} ---")
    print(doc.metadata)
    print(doc.page_content[:400].replace("\n", " "))
    print()

--- 검색 결과 1 ---
{'source': '공직자_민원응대_핵심_매뉴얼.pdf', 'page': 16, 'document_type': 'civil_complaint_manual', 'chunk_id': 29, 'chunk_size': 700, 'chunk_overlap': 120, 'chunk_strategy': 'medium', 'embedding_model': 'text-embedding-3-small', 'collection_name': 'civil_complaint_manual_medium', '_id': 'de6ff035-6ef0-45a3-859d-1560ef633e1d', '_collection_name': 'civil_complaint_manual_medium'}
질의 5 권장시간이 되면 전화를 무조건 끊어야 하는지? • 인·허가 등 정상적인 업무처리 과정에서 민원인과의 상담이 길어지는 경우는 정당한 사유없는 장시간   전화에 해당하지 않으므로 종결대상에 해당하지 않음. 질의 3 상급자 응대 후에도 계속 전화해서 폭언을 하는 경우는 어떻게 대응 해야 하는지? • 담당자에게 휴게시간 부여 등 응대를 중지하도록 조치하고 대행자 등은 다른 민원처리를 위해 해당   민원은 일시적으로 응대 보류함. • 폭언, 업무방해 등에 대해 경고문 발송, 고소·고발 등 법적 대응을 추진하여 위법행위를 지속하지   않도록 강력 대응.

--- 검색 결과 2 ---
{'source': '공직자_민원응대_핵심_매뉴얼.pdf', 'page': 16, 'document_type': 'civil_complaint_manual', 'chunk_id': 28, 'chunk_size': 700, 'chunk_overlap': 120, 'chunk_strategy': 'medium', 'embedding_model': 'text-embedding-3-small', 'collection_name': 'civil_complaint_manual_medium', '_id': '80a29

## 2. 간단한 규칙 기반 Reranking

질문 안의 핵심 키워드가 문서에 많이 포함될수록 높은 점수를 준다.

In [7]:
def simple_rerank(question: str, docs: list[Document]) -> list[Document]:
    keywords = ["권장시간", "장시간", "통화", "상담", "종료", "서면", "온라인"]

    scored = []

    for doc in docs:
        score = 0
        for keyword in keywords:
            if keyword in doc.page_content:
                score += 1
        scored.append((score, doc))

    scored.sort(key=lambda x: x[0], reverse=True)

    return [doc for score, doc in scored]


reranked_docs = simple_rerank(question, docs)

for i, doc in enumerate(reranked_docs, start=1):
    print(f"--- 재정렬 결과 {i} ---")
    print(doc.metadata)
    print(doc.page_content[:400].replace("\n", " "))
    print()

--- 재정렬 결과 1 ---
{'source': '공직자_민원응대_핵심_매뉴얼.pdf', 'page': 17, 'document_type': 'civil_complaint_manual', 'chunk_id': 30, 'chunk_size': 700, 'chunk_overlap': 120, 'chunk_strategy': 'medium', 'embedding_model': 'text-embedding-3-small', 'collection_name': 'civil_complaint_manual_medium', '_id': '9ba68a1e-3fff-4098-bc11-d1ddebaf4639', '_collection_name': 'civil_complaint_manual_medium'}
Ⅴ. 질의응답 | 3332 | 현장공무원을 위한 민원응대 핵심매뉴얼 Ⅴ 질 의 응 답 질의 6 권장시간이 지났는데도 계속 상담을 요구하는 경우에는? • 서면, 온라인 신청 등 추가 민원신청 방법을 안내.  “선생님, 전화로는 간단한 절차나 형식 등에 대해서 답변드릴 수 있습니다.   더 자세하고 정확한 민원처리를 원하시면 서면 또는 온라인으로 정식 민원접수를 해주시기 바랍니다.” 질의 7 반복전화 종료 이후에도 전화를 계속할 경우에는 어떻게 해야하는지? • 담당자에게 휴게시간 부여 등 응대를 중지하도록 조치하고 상급자 등이 응대하여 추가 설명 후 전화   종료. 그럼에도 반복전화 시에는 다른 민원처리를 위해 해당 전화는 일시적으로 응대 보류함. 02 대면 응대 질의 1 비상대응팀(반)이 구성되어 있지 않으면

--- 재정렬 결과 2 ---
{'source': '공직자_민원응대_핵심_매뉴얼.pdf', 'page': 9, 'document_type': 'civil_complaint_manual', 'chunk_id': 13, 'chunk_size': 700, 'chunk_overlap': 120, 'chunk_strategy': 'medium', 'embedding_model': '

## 3. Context Compression

검색 문단 전체를 LLM에 넣지 않고, 답변에 필요한 핵심 근거만 요약한다.

In [9]:
def format_docs(docs: list[Document]) -> str:
    return "\n\n".join([
        f"[근거 {i}] page={doc.metadata.get('page')}, chunk={doc.metadata.get('chunk_id')}\n{doc.page_content}"
        for i, doc in enumerate(docs, start=1)
    ])


context = format_docs(reranked_docs[:4])

compression_prompt = ChatPromptTemplate.from_messages([
    ("system", '''
검색된 문단에서 질문 답변에 필요한 핵심 근거만 정리하세요.
문단에 없는 내용은 추가하지 마세요.
'''),
    ("human", '''
질문:
{question}

검색 문단:
{context}

핵심 근거를 항목별로 정리하세요.
''')
])

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

compression_chain = compression_prompt | llm | StrOutputParser()

compressed_context = compression_chain.invoke({
    "question": question,
    "context": context
})

print(compressed_context)

1. **서면 또는 온라인 민원 신청 안내**: 권장시간이 지났을 경우, 추가 민원신청 방법으로 서면 또는 온라인 접수를 안내한다. 
   - 예시: “선생님, 전화로는 간단한 절차나 형식 등에 대해서 답변드릴 수 있습니다. 더 자세하고 정확한 민원처리를 원하시면 서면 또는 온라인으로 정식 민원접수를 해주시기 바랍니다.” (근거 1)

2. **통화 종료 안내**: 상담 권장시간이 초과되면 통화를 종료할 수 있으며, 다른 민원 처리를 위해 통화 종료를 안내한다.
   - 예시: “선생님, 죄송하지만 상담 권장시간이 초과되어 다른 민원 처리를 위해서 통화를 종료하오니 양해하여 주시기 바랍니다.” (근거 2)

3. **장시간 상담 시 안내**: 15분 이상 통화 시 장시간 상담이 곤란하다는 점을 안내하고, 용건 위주로 질문을 유도한다.
   - 예시: “장시간 상담 곤란 안내 및 용건 위주 질문유도” (근거 2)

4. **동일한 답변 반복 시 통화 종료 가능**: 충분한 설명 후 동일한 이야기를 반복하는 경우, 통화를 종료할 수 있다.
   - 예시: “선생님, 말씀하시는 내용에 대해서는 이미 충분히 설명을 드렸고, 동일한 답변을 드릴 수 밖에 없습니다.” (근거 3)

5. **폭언 시 통화 종료**: 지속적인 폭언이 있을 경우, 상담이 어렵다는 이유로 통화를 종료할 수 있다.
   - 예시: “지속적인 폭언 등으로 더 이상 상담이 어렵습니다. 통화를 종료하겠습니다.” (근거 4)


## 4. 압축된 근거로 답변 생성

In [10]:
answer_prompt = ChatPromptTemplate.from_messages([
    ("system", '''
당신은 공직자 민원응대 매뉴얼 기반 업무지원 AI입니다.
제공된 근거를 바탕으로만 답변하세요.

답변 형식:
1. 핵심 대응
2. 단계별 조치
3. 안내 표현
4. 주의사항
'''),
    ("human", '''
질문:
{question}

핵심 근거:
{context}
''')
])

answer_chain = answer_prompt | llm | StrOutputParser()

answer = answer_chain.invoke({
    "question": question,
    "context": compressed_context
})

print(answer)

1. **핵심 대응**: 권장시간이 초과된 경우, 서면 또는 온라인 민원 신청을 안내하고 통화를 종료할 수 있습니다.

2. **단계별 조치**:
   - 상담 권장시간이 초과되었음을 알립니다.
   - 서면 또는 온라인 민원 신청 방법을 안내합니다.
   - 통화를 종료할 필요가 있을 경우, 다른 민원 처리를 위해 종료할 것임을 설명합니다.

3. **안내 표현**:
   - “선생님, 전화로는 간단한 절차나 형식 등에 대해서 답변드릴 수 있습니다. 더 자세하고 정확한 민원처리를 원하시면 서면 또는 온라인으로 정식 민원접수를 해주시기 바랍니다.”
   - “선생님, 죄송하지만 상담 권장시간이 초과되어 다른 민원 처리를 위해서 통화를 종료하오니 양해하여 주시기 바랍니다.”

4. **주의사항**:
   - 상담 중 동일한 내용을 반복하는 경우, 충분한 설명 후 통화를 종료할 수 있음을 유의해야 합니다.
   - 폭언이 지속될 경우, 상담이 어렵다는 이유로 통화를 종료할 수 있음을 명확히 해야 합니다.


## 핵심 정리

- Reranking은 검색 결과의 순서를 다시 조정하는 과정이다.
- Context Compression은 답변에 필요한 근거만 줄여서 LLM에 전달하는 과정이다.
- Modular RAG에서는 `retrieve`, `rerank`, `compress`, `generate`를 각각 분리할 수 있다.